# Pipeline Tests
Covers:
1. **Synthetic data helpers** — shared across all tests
2. **Minimal model** — bare-bones `BaseModel` subclass
3. **`train()` — concat mixing** (default, single dataset)
4. **`train()` — multi-dataset concat** (two datasets, weighted mixing)
5. **`train()` — round-robin mixing** (two horizons)
6. **`train()` — sharded dataset** (`write_sharded_dataset` + `ShardedTrainDataset`)
7. **`train_distributed()` — sharded, mp.spawn** (2 ranks, CPU/NCCL auto-detected)
8. **`fork_sequences` shape assertions** — standalone unit tests

## 0. Imports & path setup

In [ ]:
import sys, os, shutil, tempfile, warnings
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

NOTEBOOK_DIR = Path().resolve()                    # tests/
SRC_DIR      = NOTEBOOK_DIR.parent / "src"         # src/

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# ── Imports ───────────────────────────────────────────────────────────────────
from dataloaders._forking_sequences import fork_sequences
from dataloaders.ts_dataloader      import DataLoaderFactory, FullSeriesDataset
from dataloaders.ts_sharding        import (
    write_sharded_dataset,
    ShardedTrainDataset,
    ShardedValDataset,
    ShardedTestDataset,
)
from common._base_model import BaseModel
from common.train       import train, train_distributed, eval_test

warnings.filterwarnings("ignore")
print("imports OK")


## 1. Shared helpers

In [ ]:
# ── Synthetic dataframe factory ───────────────────────────────────────────────

def make_df(
    n_series: int = 3,
    n_steps:  int = 500,
    n_hist:   int = 1,       # number of hist exog cols
    seed:     int = 42,
) -> pd.DataFrame:
    """
    Returns a long-format DataFrame with columns:
        unique_id, ds, y, available_mask, hist_0, hist_1, ...
    All series have equal length (n_steps timesteps).
    available_mask is 1 everywhere (no gaps).
    """
    rng   = np.random.default_rng(seed)
    times = pd.date_range("2020-01-01", periods=n_steps, freq="5min")
    rows  = []
    for s in range(n_series):
        y    = rng.standard_normal(n_steps).astype(np.float32)
        hist = {f"hist_{i}": rng.standard_normal(n_steps).astype(np.float32)
                for i in range(n_hist)}
        df_s = pd.DataFrame({"unique_id": f"series_{s}", "ds": times, "y": y,
                              "available_mask": np.ones(n_steps, dtype=np.float32),
                              **hist})
        rows.append(df_s)
    return pd.concat(rows, ignore_index=True)


# ── Config factories ──────────────────────────────────────────────────────────

def make_mcfg(
    context_length:    int  = 64,
    fcd_samples:       int  = 4,
    batch_size:        int  = 2,
    max_steps:         int  = 20,
    val_check_interval:int  = 10,
    mixing_strategy:   str  = "concat",
    normalize:         bool = False,
    checkpoint_dir:    str  = "/tmp/ckpts",
    num_workers:       int  = 0,
):
    return SimpleNamespace(
        context_length         = context_length,
        input_size             = context_length,
        fcd_samples            = fcd_samples,
        batch_size             = batch_size,
        valid_batch_size       = batch_size,
        max_steps              = max_steps,
        val_check_interval     = val_check_interval,
        learning_rate          = 1e-3,
        gradient_clip_val      = 1.0,
        early_stopping_patience= 9999,  # disable for tests
        monitor_metric         = "loss",
        monitor_mode           = "min",
        mixing_strategy        = mixing_strategy,
        drop_last              = False,
        normalize              = normalize,
        batch_mode             = "full_series",
        checkpoint_dir         = checkpoint_dir,
        checkpoint_step        = 99999,  # suppress mid-run saves
        num_workers            = num_workers,
    )


def make_entry(
    path:            str,
    name:            str  = "ds",
    horizon:         int  = 6,
    val_size:        int  = 50,
    test_size:       int  = 50,
    weight:          float= 1.0,
    hist_exog_cols:  list = None,
    futr_exog_cols:  list = None,
    stat_exog_cols:  list = None,
    per_series_split:bool = False,
    use_context_head:bool = True,
    sharded_dir:     str  = None,
):
    return SimpleNamespace(
        path             = path,
        name             = name,
        horizon          = horizon,
        val_size         = val_size,
        test_size        = test_size,
        weight           = weight,
        hist_exog_cols   = hist_exog_cols  or [],
        futr_exog_cols   = futr_exog_cols  or [],
        stat_exog_cols   = stat_exog_cols  or [],
        per_series_split = per_series_split,
        use_context_head = use_context_head,
        sharded_dir      = sharded_dir,
    )


def make_dcfg(train_entries, val_entries=None, test_entries=None):
    return SimpleNamespace(
        train      = train_entries,
        validation = val_entries  or train_entries,
        test       = test_entries or train_entries,
    )


print("helpers OK")

## 2. Minimal model

In [ ]:
class TinyLinearModel(BaseModel):
    """
    Simplest possible BaseModel subclass for pipeline testing.

    forward: flatten insample_y → linear → reshape to [B*n_fcds, H, C]
    compute_loss: MSE vs outsample_y (inherited default).

    Not useful for forecasting — only here to exercise the pipeline end-to-end.
    """
    def __init__(self, context_length: int, horizon: int, n_channels: int,
                 n_hist: int = 1):
        super().__init__()
        in_features  = context_length * n_channels * (1 + n_hist)
        out_features = horizon * n_channels
        self.fc = nn.Linear(in_features, out_features)
        self._ctx = context_length
        self._H   = horizon
        self._C   = n_channels

    def forward(self, batch):
        # insample_y : [B, enc_size, C, 1+Vh]
        # outsample_y: [B, n_fcds, H, C]
        x       = batch["insample_y"]                # [B, enc_size, C, 1+Vh]
        B       = x.shape[0]
        n_fcds  = batch["outsample_y"].shape[1]

        # Take only the last context_length steps, flatten per FCD
        x_ctx   = x[:, -self._ctx:, :, :]           # [B, L, C, 1+Vh]
        # Repeat for each FCD (naive — just for shape correctness)
        x_rep   = x_ctx.unsqueeze(1).expand(B, n_fcds, -1, -1, -1)  # [B, n_fcds, L, C, 1+Vh]
        flat    = x_rep.reshape(B * n_fcds, -1)     # [B*n_fcds, L*C*(1+Vh)]

        # Pad/trim to expected in_features
        expected = self.fc.in_features
        if flat.shape[1] < expected:
            flat = torch.nn.functional.pad(flat, (0, expected - flat.shape[1]))
        else:
            flat = flat[:, :expected]

        out = self.fc(flat)                          # [B*n_fcds, H*C]
        return out.reshape(B, n_fcds, self._H, self._C)


def make_model(mcfg, n_channels=3, n_hist=0):
    return TinyLinearModel(
        context_length = mcfg.context_length,
        horizon        = 6,
        n_channels     = n_channels,
        n_hist         = n_hist,
    )


print("model OK")

---
## Test 1 — `train()`, single dataset, concat mixing (default)

In [ ]:
print("=" * 60)
print("TEST 1 — single dataset, concat mixing")
print("=" * 60)

with tempfile.TemporaryDirectory() as tmpdir:
    csv_path = f"{tmpdir}/data.csv"
    df = make_df(n_series=3, n_steps=300)
    df.to_csv(csv_path, index=False)

    mcfg = make_mcfg(
        context_length = 32,
        fcd_samples    = 4,
        batch_size     = 2,
        max_steps      = 20,
        mixing_strategy= "concat",
        checkpoint_dir = tmpdir,
    )
    entry = make_entry(csv_path, name="test", horizon=6, val_size=50, test_size=50)
    dcfg  = make_dcfg([entry])

    factory      = DataLoaderFactory(mcfg, dcfg)
    train_loader = factory.train_dataloader()
    val_loaders  = factory.val_dataloaders()

    model   = make_model(mcfg, n_channels=3)
    metrics = train(model, mcfg, train_loader, val_loaders,
                    device=torch.device("cpu"), seed=0)

    print(f"\nfinal val metrics: {metrics}")
    assert Path(tmpdir, "final.pt").exists(), "final.pt not saved"

    # Check predictions
    test_loaders = factory.test_dataloaders()
    preds = eval_test(model, factory, device=torch.device("cpu"))
    assert "test" in preds, "missing test predictions"

print("\n✓ TEST 1 PASSED")

---
## Test 2 — `train()`, two datasets, weighted concat mixing

In [ ]:
print("=" * 60)
print("TEST 2 — two datasets, weighted concat mixing")
print("=" * 60)

with tempfile.TemporaryDirectory() as tmpdir:
    csv_a = f"{tmpdir}/data_a.csv"
    csv_b = f"{tmpdir}/data_b.csv"
    make_df(n_series=3, n_steps=300, seed=1).to_csv(csv_a, index=False)
    make_df(n_series=2, n_steps=300, seed=2).to_csv(csv_b, index=False)

    mcfg = make_mcfg(
        context_length  = 32,
        fcd_samples     = 4,
        batch_size      = 2,
        max_steps       = 20,
        mixing_strategy = "concat",
        checkpoint_dir  = tmpdir,
    )

    # Same horizon, different weights — dataset A gets 3x more samples than B
    entry_a = make_entry(csv_a, name="ds_a", horizon=6, val_size=50, test_size=50, weight=3.0)
    entry_b = make_entry(csv_b, name="ds_b", horizon=6, val_size=50, test_size=50, weight=1.0)
    dcfg    = make_dcfg([entry_a, entry_b])

    factory      = DataLoaderFactory(mcfg, dcfg)
    train_loader = factory.train_dataloader()
    val_loaders  = factory.val_dataloaders()

    # Max channels across both datasets = 3 (ds_a)
    model   = make_model(mcfg, n_channels=3)
    metrics = train(model, mcfg, train_loader, val_loaders,
                    device=torch.device("cpu"), seed=0)

    print(f"final val metrics: {metrics}")

    # Verify both datasets appear in val/test
    preds = eval_test(model, factory, device=torch.device("cpu"))
    assert "ds_a" in preds and "ds_b" in preds, f"missing keys in preds: {preds.keys()}"

print("\n✓ TEST 2 PASSED")

In [ ]:
preds.keys()

---
## Test 4 — `train()`, sharded dataset

In [ ]:
print("=" * 60)
print("TEST 4 — sharded dataset (write → train → val → test)")
print("=" * 60)

with tempfile.TemporaryDirectory() as tmpdir:
    shard_dir = f"{tmpdir}/sharded"
    csv_path  = f"{tmpdir}/data.csv"

    VAL_SIZE  = 60
    TEST_SIZE = 60
    CTX       = 32
    HORIZON   = 6

    df = make_df(n_series=3, n_steps=400, n_hist=1, seed=42)
    df.to_csv(csv_path, index=False)

    # ── Write shards ──────────────────────────────────────────────────────────
    write_sharded_dataset(
        df             = df,
        out_dir        = shard_dir,
        val_size       = VAL_SIZE,
        test_size      = TEST_SIZE,
        context_length = CTX,
        shard_size     = 100,   # small so we get multiple shards
        hist_exog_cols = ["hist_0"],
    )

    shard_files = sorted(Path(shard_dir).glob("shard_*.parquet"))
    print(f"wrote {len(shard_files)} train shard(s) + val.parquet + test.parquet")
    assert (Path(shard_dir) / "val.parquet").exists(),  "val.parquet missing"
    assert (Path(shard_dir) / "test.parquet").exists(), "test.parquet missing"

    # ── Check split sizes ─────────────────────────────────────────────────────
    import json
    meta = json.loads((Path(shard_dir) / "metadata.json").read_text())
    assert meta["T_val"]  == VAL_SIZE,  f"T_val mismatch: {meta['T_val']}"
    assert meta["T_test"] == TEST_SIZE, f"T_test mismatch: {meta['T_test']}"
    T_train_expected = 400 - VAL_SIZE - TEST_SIZE
    assert meta["T_train"] == T_train_expected, f"T_train mismatch: {meta['T_train']}"
    print(f"split sizes: T_train={meta['T_train']}  T_val={meta['T_val']}  T_test={meta['T_test']}")

    # ── Check context heads ───────────────────────────────────────────────────
    val_df  = pd.read_parquet(Path(shard_dir) / "val.parquet")
    test_df = pd.read_parquet(Path(shard_dir) / "test.parquet")
    assert len(val_df)  == VAL_SIZE  + CTX, f"val file length wrong: {len(val_df)}"
    assert len(test_df) == TEST_SIZE + CTX, f"test file length wrong: {len(test_df)}"
    print(f"val.parquet  rows: {len(val_df)}  (= {VAL_SIZE} + {CTX} context)")
    print(f"test.parquet rows: {len(test_df)} (= {TEST_SIZE} + {CTX} context)")

    # ── Train via factory ─────────────────────────────────────────────────────
    mcfg = make_mcfg(
        context_length = CTX,
        fcd_samples    = 4,
        batch_size     = 2,
        max_steps      = 20,
        checkpoint_dir = tmpdir,
    )
    entry = make_entry(
        path           = csv_path,    # still needed for test fallback
        name           = "sharded_ds",
        horizon        = HORIZON,
        val_size       = VAL_SIZE,
        test_size      = TEST_SIZE,
        hist_exog_cols = ["hist_0"],
        sharded_dir    = shard_dir,   # triggers sharded path
    )
    dcfg = make_dcfg([entry])

    factory      = DataLoaderFactory(mcfg, dcfg)
    train_loader = factory.train_dataloader()
    val_loaders  = factory.val_dataloaders()

    # Verify ShardedTrainDataset is loaded (rank=0, all shards assigned)
    ds = list(train_loader.dataset.datasets)[0]
    assert isinstance(ds, ShardedTrainDataset), f"expected ShardedTrainDataset, got {type(ds)}"
    print(f"ShardedTrainDataset: T={ds.T}  n_windows={ds._n_windows}")

    ds_val = list(factory.val_dataloaders().values())[0].dataset
    assert isinstance(ds_val, ShardedValDataset), f"expected ShardedValDataset, got {type(ds_val)}"
    print(f"ShardedValDataset:   T={ds_val.T}")

    model   = TinyLinearModel(context_length=CTX, horizon=HORIZON, n_channels=3, n_hist=1)
    metrics = train(model, mcfg, train_loader, val_loaders,
                    device=torch.device("cpu"), seed=0)
    print(f"val metrics: {metrics}")

    # Test set uses ShardedTestDataset
    preds = eval_test(model, factory, device=torch.device("cpu"))
    ds_test_loaded = list(factory.test_dataloaders().values())[0].dataset
    assert isinstance(ds_test_loaded, ShardedTestDataset), f"expected ShardedTestDataset"
    print(f"ShardedTestDataset:  T={ds_test_loaded.T}")
    print(f"test pred shape: {tuple(preds['sharded_ds'].shape)}")

print("\n✓ TEST 4 PASSED")

---
## Test 6 — `train()`, resume from checkpoint

In [ ]:
print("=" * 60)
print("TEST 6 — resume training from checkpoint")
print("=" * 60)

with tempfile.TemporaryDirectory() as tmpdir:
    csv_path = f"{tmpdir}/data.csv"
    make_df(n_series=3, n_steps=300).to_csv(csv_path, index=False)

    mcfg = make_mcfg(
        context_length=32, fcd_samples=4, batch_size=2,
        max_steps=10, checkpoint_dir=tmpdir,
    )
    entry = make_entry(csv_path, name="resume_ds", horizon=6, val_size=50, test_size=50)
    dcfg  = make_dcfg([entry])

    # First run — saves final.pt
    factory      = DataLoaderFactory(mcfg, dcfg)
    train_loader = factory.train_dataloader()
    val_loaders  = factory.val_dataloaders()
    model = make_model(mcfg)
    train(model, mcfg, train_loader, val_loaders, device=torch.device("cpu"))
    step_after_first = model.global_step
    print(f"after first run: global_step={step_after_first}")

    # Second run — resume from final.pt, train 10 more steps
    mcfg2 = make_mcfg(
        context_length=32, fcd_samples=4, batch_size=2,
        max_steps=step_after_first + 10, checkpoint_dir=tmpdir,
    )
    factory2     = DataLoaderFactory(mcfg2, dcfg)
    model2       = make_model(mcfg2)
    train(
        model2, mcfg2,
        factory2.train_dataloader(), factory2.val_dataloaders(),
        device  = torch.device("cpu"),
        resume  = f"{tmpdir}/final.pt",
    )
    print(f"after resume:    global_step={model2.global_step}")
    assert model2.global_step == step_after_first + 10, \
        f"expected {step_after_first + 10}, got {model2.global_step}"

print("\n✓ TEST 6 PASSED")

---
## Test 7 — `train_distributed()`, sharded dataset, mp.spawn

In [ ]:
print("=" * 60)
print("TEST 7 — train_distributed (mp.spawn, 2 ranks, sharded)")
print("=" * 60)

# Use gloo backend so this works on CPU / machines without NCCL
BACKEND    = "nccl" if torch.cuda.is_available() else "gloo"
WORLD_SIZE = min(2, torch.cuda.device_count() or 2)
print(f"backend={BACKEND}  world_size={WORLD_SIZE}")

with tempfile.TemporaryDirectory() as tmpdir:
    shard_dir = f"{tmpdir}/sharded"
    csv_path  = f"{tmpdir}/data.csv"

    CTX     = 32
    HORIZON = 6

    df = make_df(n_series=3, n_steps=600, seed=99)
    df.to_csv(csv_path, index=False)

    write_sharded_dataset(
        df             = df,
        out_dir        = shard_dir,
        val_size       = 80,
        test_size      = 80,
        context_length = CTX,
        shard_size     = 100,   # small shards — ensures each rank gets ≥1 shard
    )

    shard_files = sorted(Path(shard_dir).glob("shard_*.parquet"))
    print(f"{len(shard_files)} train shards written")
    assert len(shard_files) >= WORLD_SIZE, \
        f"need ≥{WORLD_SIZE} shards for {WORLD_SIZE} ranks, got {len(shard_files)}"

    mcfg = make_mcfg(
        context_length  = CTX,
        fcd_samples     = 4,
        batch_size      = 2,
        max_steps       = 20,
        val_check_interval = 10,
        checkpoint_dir  = tmpdir,
    )
    entry = make_entry(
        path        = csv_path,
        name        = "dist_ds",
        horizon     = HORIZON,
        val_size    = 80,
        test_size   = 80,
        sharded_dir = shard_dir,
    )
    dcfg = make_dcfg([entry])

    # Factory is created BEFORE spawn — shared config, datasets rebuilt per rank
    factory = DataLoaderFactory(mcfg, dcfg)
    model   = TinyLinearModel(context_length=CTX, horizon=HORIZON, n_channels=3)

    train_distributed(
        model      = model,
        mcfg       = mcfg,
        factory    = factory,
        backend    = BACKEND,
        use_spawn  = True,
        world_size = WORLD_SIZE,
        seed       = 42,
    )

    assert Path(tmpdir, "final.pt").exists(), "final.pt not saved by rank 0"
    print("final.pt saved ✓")

    # Load weights and run test on single GPU
    test_model = TinyLinearModel(context_length=CTX, horizon=HORIZON, n_channels=3)
    BaseModel.load_weights(f"{tmpdir}/final.pt", test_model)

    # Rebuild factory for test (no distributed context)
    factory_test = DataLoaderFactory(mcfg, dcfg)
    preds = eval_test(test_model, factory_test, device=torch.device("cpu"))
    print(f"test pred shape: {tuple(preds['dist_ds'].shape)}")

print("\n✓ TEST 7 PASSED")

---
## Test 8 — `fork_sequences` unit tests

In [ ]:
print("=" * 60)
print("TEST 8 — fork_sequences shape assertions")
print("=" * 60)

def make_batch(B=2, T=200, C=3, Vh=1, H=6, device="cpu"):
    """Minimal collated batch as _full_series_collate_fn would produce."""
    return {
        "x_enc":          torch.randn(B, T, C, 1 + Vh, device=device),
        "available_mask": torch.ones(B, C, T, device=device),
        "channel_mask":   torch.ones(B, C, device=device),
        "horizon":        torch.full((B,), H, dtype=torch.long, device=device),
    }

L, H, S = 32, 6, 1

# ── Training path (fcd_samples=4) ─────────────────────────────────────────────
print("\n--- Training path (fcd_samples=4) ---")
batch = make_batch(B=2, T=200, C=3, Vh=1, H=H)
out   = fork_sequences(batch, context_length=L, fcd_samples=4, horizon=H)

B_out, enc_size, C_out, feat = out["insample_y"].shape
print(f"insample_y:    {tuple(out['insample_y'].shape)}    expected [2, enc_size, 3, 2]")
print(f"outsample_y:   {tuple(out['outsample_y'].shape)}   expected [2, 4, {H}, 3]")
print(f"available_mask:{tuple(out['available_mask'].shape)}")

assert out["outsample_y"].shape == (2, 4, H, 3), \
    f"outsample_y shape wrong: {out['outsample_y'].shape}"
assert out["available_mask"].shape == (2, 3, enc_size), \
    f"available_mask shape wrong: {out['available_mask'].shape}"
print("training path shapes ✓")

# ── Val/test path (fcd_samples=-1) ────────────────────────────────────────────
print("\n--- Val/test path (fcd_samples=-1) ---")
T = 200
batch_val  = make_batch(B=2, T=T, C=3, Vh=0, H=H)
out_val    = fork_sequences(batch_val, context_length=L, fcd_samples=-1, horizon=H)
n_expected = n_valid_fcds(T, L, H, S)

print(f"n_valid_fcds({T}, L={L}, H={H}, S={S}) = {n_expected}")
print(f"outsample_y:   {tuple(out_val['outsample_y'].shape)}  expected [2, {n_expected}, {H}, 3]")
assert out_val["outsample_y"].shape[1] == n_expected, \
    f"n_fcds wrong: {out_val['outsample_y'].shape[1]} != {n_expected}"
print("val/test path shapes ✓")

# ── Left-padding: windows never start in padding ───────────────────────────────
print("\n--- Left-padding: sampler skips padded timesteps ---")
T_real, T_pad = 100, 50
T_total       = T_real + T_pad
mask = torch.zeros(1, 3, T_total)
mask[:, :, T_pad:] = 1.0   # real data starts at T_pad

batch_pad = {
    "x_enc":          torch.randn(1, T_total, 3, 1),
    "available_mask": mask,
    "horizon":        torch.tensor([H]),
}
from fork_sequences import heterogeneous_sampler
for _ in range(20):
    ws, _ = heterogeneous_sampler(mask, L, fcd_samples=4, horizon=H)
    assert ws[0].item() >= T_pad, \
        f"window_start {ws[0].item()} < T_pad {T_pad} — sampled in padding!"
print("all 20 sampled window_starts ≥ T_pad ✓")

# ── available_mask shape assertion ────────────────────────────────────────────
print("\n--- available_mask shape mismatch raises AssertionError ---")
bad_mask_batch = {
    "x_enc":          torch.randn(2, 100, 3, 1),
    "available_mask": torch.ones(2, 100),   # wrong: [B, T] instead of [B, C, T]
    "horizon":        torch.tensor([H, H]),
}
try:
    fork_sequences(bad_mask_batch, context_length=L, fcd_samples=4, horizon=H)
    assert False, "should have raised"
except AssertionError as e:
    print(f"raised AssertionError as expected: {e}")

print("\n✓ TEST 8 PASSED")

---
## Test 9 — distributed val sanity check (loss averaged across ranks)

In [ ]:
print("=" * 60)
print("TEST 9 — distributed val: each rank evaluates full val set,")
print("         global loss = mean of per-rank losses")
print("=" * 60)

# We verify this without actually spawning by simulating what each rank
# would compute: load val.parquet, run fork_sequences(-1), compute MSE.
# The 'global' loss should equal the mean because every rank sees
# identical data (no DistributedSampler).

with tempfile.TemporaryDirectory() as tmpdir:
    shard_dir = f"{tmpdir}/sharded"
    df = make_df(n_series=3, n_steps=400)
    write_sharded_dataset(
        df, shard_dir, val_size=60, test_size=60,
        context_length=32, shard_size=100,
    )

    # Simulate two "ranks" — same model weights, same val data
    ds_val = ShardedValDataset(shard_dir, context_length=32, horizon=6)
    item   = ds_val[0]

    # Both ranks would compute identical loss since weights and data are the same
    model_r0 = TinyLinearModel(context_length=32, horizon=6, n_channels=3)
    model_r1 = TinyLinearModel(context_length=32, horizon=6, n_channels=3)
    # Copy weights so they are identical (simulates fresh DDP init)
    model_r1.load_state_dict(model_r0.state_dict())

    mcfg = make_mcfg(context_length=32, fcd_samples=4,
                     batch_size=1, max_steps=5, checkpoint_dir=tmpdir)

    from torch.utils.data import DataLoader
    from ts_dataloader import _full_series_collate_fn

    val_loader = DataLoader(ds_val, batch_size=1, collate_fn=_full_series_collate_fn)

    def rank_val_loss(model):
        model.eval()
        total, n = 0.0, 0
        with torch.no_grad():
            for batch in val_loader:
                fcd_batch = fork_sequences(
                    batch, context_length=32, fcd_samples=-1,
                    horizon=int(batch["horizon"][0].item())
                )
                pred = model(fcd_batch)
                loss = model.loss_fn(
                    pred,
                    fcd_batch["outsample_y"].reshape(-1, 6, 3)
                )
                total += loss.item()
                n     += 1
        return total / n

    # Setup a minimal loss_fn on the models
    for m in [model_r0, model_r1]:
        m.loss_fn = torch.nn.functional.mse_loss

    loss_r0 = rank_val_loss(model_r0)
    loss_r1 = rank_val_loss(model_r1)

    # Since weights are identical, losses should be equal
    global_loss = (loss_r0 + loss_r1) / 2
    print(f"rank 0 val loss:  {loss_r0:.6f}")
    print(f"rank 1 val loss:  {loss_r1:.6f}")
    print(f"global mean loss: {global_loss:.6f}")
    assert abs(loss_r0 - loss_r1) < 1e-5, "identical models should get identical val loss"
    assert abs(global_loss - loss_r0) < 1e-5

print("\n✓ TEST 9 PASSED")

---
## Summary

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════╗
║                    All tests complete                        ║
╠══════════════════════════════════════════════════════════════╣
║  1  single dataset, concat mixing                            ║
║  2  two datasets, weighted concat mixing                     ║
║  3  round-robin mixing, two horizons                         ║
║  4  sharded dataset (write → train → val → test)             ║
║  5  hist exog features + normalisation                       ║
║  6  resume from checkpoint                                   ║
║  7  train_distributed (mp.spawn, sharded, 2 ranks)           ║
║  8  fork_sequences unit tests                                ║
║  9  distributed val: global loss = mean of per-rank losses   ║
╚══════════════════════════════════════════════════════════════╝
""")